# 02 – Data Exploration

Erste Übersicht über alle Polar-Gesundheitsdaten: Vollständigkeit, Verteilungen,
zeitliche Abdeckung und erste visuelle Eindrücke.

## Inhalt
1. Setup & Daten laden
2. Datenqualität & Vollständigkeit
3. Activity – Schritte, Kalorien, Schlaf
4. Herzfrequenz – Ruhepuls-Entwicklung
5. Training – Sportarten & Volumen
6. HRV – Überblick
7. Kombinierte Übersicht

## 1 – Setup & Daten laden

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from db_loader import DatabricksLoader, secrets_pruefen

secrets_pruefen()

# ── Plot-Stil ────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize'  : (14, 4),
    'figure.dpi'      : 120,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 9,
    'ytick.labelsize' : 9,
    'axes.spines.top' : False,
    'axes.spines.right': False,
})

# ── Daten laden ──────────────────────────────────────────────
db = DatabricksLoader()

df_act   = db.lade_activity()
df_train = db.lade_training()
df_hr    = db.lade_heartrate()
df_hrv   = db.lade_hrv()

# Datum als datetime für Zeitreihen
for df in [df_act, df_train, df_hr, df_hrv]:
    if not df.empty:
        df['datum'] = pd.to_datetime(df['datum'])

print("\n✅ Alle Daten geladen")

## 2 – Datenqualität & Vollständigkeit

In [ ]:
# ── Zeitliche Abdeckung ──────────────────────────────────────
print("═" * 55)
print("  DATENÜBERSICHT – Polar Health Analytics")
print("═" * 55)

tabellen = [
    ('Activity',      df_act,   'schritte'),
    ('Herzfrequenz',  df_hr,    'hr_ruhepuls'),
    ('Training',      df_train, 'dauer_min'),
    ('HRV',           df_hrv,   'hrv_rmssd'),
]

for name, df, kern_spalte in tabellen:
    if df.empty:
        print(f"  {name:<14}: ⚠️  Keine Daten")
        continue
    von   = df['datum'].min().strftime('%d.%m.%Y')
    bis   = df['datum'].max().strftime('%d.%m.%Y')
    jahre = (df['datum'].max() - df['datum'].min()).days / 365.25
    nulls = df[kern_spalte].isna().sum()
    null_pct = 100 * nulls / len(df)
    print(f"  {name:<14}: {len(df):>5} Zeilen  │  {von} – {bis}  "
          f"({jahre:.1f} Jahre)  │  {kern_spalte} NaN: {null_pct:.1f}%")

print("═" * 55)

In [ ]:
# ── Fehlende Tage visualisieren (Activity) ───────────────────
if not df_act.empty:
    alle_tage = pd.date_range(df_act['datum'].min(), df_act['datum'].max(), freq='D')
    fehlende  = alle_tage.difference(df_act['datum'])
    pct_abdeckung = 100 * (1 - len(fehlende) / len(alle_tage))

    print(f"Activity-Abdeckung: {pct_abdeckung:.1f}%  "
          f"({len(fehlende)} fehlende Tage von {len(alle_tage)})")

    # Fehlende Tage pro Jahr
    df_fehlend = pd.DataFrame({'datum': fehlende})
    if not df_fehlend.empty:
        df_fehlend['jahr'] = df_fehlend['datum'].dt.year
        print("\nFehlende Tage pro Jahr:")
        print(df_fehlend.groupby('jahr').size().to_string())

In [ ]:
# ── Statistische Übersicht ───────────────────────────────────
print("\n── Activity: Statistische Kennzahlen ──")
if not df_act.empty:
    display(df_act[['schritte','kalorien','schlaf_stunden',
                    'schlaf_qualitaet','met_minuten']].describe().round(2))

print("\n── Herzfrequenz: Statistische Kennzahlen ──")
if not df_hr.empty:
    display(df_hr[['hr_ruhepuls','hr_mean','hr_max']].describe().round(2))

print("\n── Training: Statistische Kennzahlen ──")
if not df_train.empty:
    display(df_train[['dauer_min','hr_avg','hr_max',
                       'distanz_km','kalorien']].describe().round(2))

## 3 – Activity: Schritte, Kalorien, Schlaf

In [ ]:
if not df_act.empty:
    df_m = df_act.set_index('datum').resample('ME').agg({
        'schritte'       : 'mean',
        'kalorien'       : 'mean',
        'schlaf_stunden' : 'mean',
        'schlaf_qualitaet': 'mean',
    }).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Activity – Monatsdurchschnitte', fontsize=14, y=1.02)

    # Schritte
    ax = axes[0]
    ax.plot(df_m['datum'], df_m['schritte'], color='#4C72B0', lw=1.2, alpha=0.7)
    ax.axhline(10000, color='#DD8452', lw=1.5, ls='--', label='10\'000 Ziel')
    ax.fill_between(df_m['datum'], df_m['schritte'], alpha=0.15, color='#4C72B0')
    ax.set_title('Ø Schritte pro Tag')
    ax.set_ylabel('Schritte')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # Schlaf Stunden
    ax = axes[1]
    ax.plot(df_m['datum'], df_m['schlaf_stunden'], color='#55A868', lw=1.2, alpha=0.7)
    ax.axhline(7.5, color='#DD8452', lw=1.5, ls='--', label='7.5h Empfehlung')
    ax.fill_between(df_m['datum'], df_m['schlaf_stunden'], alpha=0.15, color='#55A868')
    ax.set_title('Ø Schlafdauer (h)')
    ax.set_ylabel('Stunden')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # Schlafqualität
    ax = axes[2]
    ax.plot(df_m['datum'], df_m['schlaf_qualitaet'], color='#8172B2', lw=1.2, alpha=0.7)
    ax.fill_between(df_m['datum'], df_m['schlaf_qualitaet'], alpha=0.15, color='#8172B2')
    ax.set_title('Ø Schlafqualität (0–1)')
    ax.set_ylabel('Qualität')
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    plt.tight_layout()
    plt.show()

    # Jahres-Durchschnitte
    print("\nSchritte-Durchschnitt pro Jahr:")
    df_act['jahr'] = df_act['datum'].dt.year
    print(df_act.groupby('jahr')['schritte'].mean().round(0).to_string())

## 4 – Herzfrequenz: Ruhepuls-Entwicklung

In [ ]:
if not df_hr.empty:
    # Ruhepuls-Zeitreihe mit Glättung
    df_hr_s = df_hr.sort_values('datum').copy()
    df_hr_s['trend_90d'] = (
        df_hr_s['hr_ruhepuls']
        .rolling(90, min_periods=14, center=True).mean()
    )
    df_hr_s['trend_365d'] = (
        df_hr_s['hr_ruhepuls']
        .rolling(365, min_periods=30, center=True).mean()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    fig.suptitle('Herzfrequenz – Ruhepuls', fontsize=14, y=1.02)

    # Zeitreihe
    ax = axes[0]
    ax.scatter(df_hr_s['datum'], df_hr_s['hr_ruhepuls'],
               s=1.5, alpha=0.3, color='#4C72B0', label='Tagesmessung')
    ax.plot(df_hr_s['datum'], df_hr_s['trend_90d'],
            color='#DD8452', lw=2, label='90-Tage-Trend')
    ax.plot(df_hr_s['datum'], df_hr_s['trend_365d'],
            color='#C44E52', lw=2.5, ls='--', label='365-Tage-Trend')
    ax.set_title('Ruhepuls-Entwicklung (2014–2026)')
    ax.set_ylabel('Ruhepuls (bpm)')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # Verteilung
    ax = axes[1]
    ax.hist(df_hr_s['hr_ruhepuls'].dropna(), bins=40,
            color='#4C72B0', alpha=0.75, edgecolor='white')
    median_hr = df_hr_s['hr_ruhepuls'].median()
    ax.axvline(median_hr, color='#DD8452', lw=2,
               label=f'Median: {median_hr:.0f} bpm')
    ax.set_title('Verteilung Ruhepuls')
    ax.set_xlabel('Ruhepuls (bpm)')
    ax.set_ylabel('Häufigkeit')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Jahres-Durchschnitte
    df_hr['jahr'] = df_hr['datum'].dt.year
    print("\nØ Ruhepuls pro Jahr:")
    print(df_hr.groupby('jahr')['hr_ruhepuls'].mean().round(1).to_string())

In [ ]:
# ── Ruhepuls nach Wochentag (Boxplot) ───────────────────────
if not df_hr.empty:
    wochentage = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
    df_hr['wochentag_kurz'] = df_hr['wochentag_nr'].map(
        dict(enumerate(wochentage))
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.boxplot(
        data=df_hr,
        x='wochentag_kurz', y='hr_ruhepuls',
        order=wochentage,
        palette='Blues',
        width=0.5, ax=ax
    )
    ax.set_title('Ruhepuls nach Wochentag')
    ax.set_xlabel('Wochentag')
    ax.set_ylabel('Ruhepuls (bpm)')
    plt.tight_layout()
    plt.show()

## 5 – Training: Sportarten & Volumen

In [ ]:
if not df_train.empty:
    # Top-Sportarten
    top_sports = df_train['sport'].value_counts().head(10)
    print("Top 10 Sportarten:")
    print(top_sports.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Training – Überblick', fontsize=14, y=1.02)

    # Trainings pro Jahr
    df_train['jahr'] = df_train['datum'].dt.year
    trainings_jahr = df_train.groupby('jahr').size()
    ax = axes[0]
    bars = ax.bar(trainings_jahr.index, trainings_jahr.values,
                  color='#4C72B0', alpha=0.8, edgecolor='white')
    ax.set_title('Trainingseinheiten pro Jahr')
    ax.set_xlabel('Jahr')
    ax.set_ylabel('Anzahl')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                str(int(h)), ha='center', va='bottom', fontsize=7)

    # Sportarten-Donut
    ax = axes[1]
    top5 = df_train['sport'].value_counts().head(5)
    andere = len(df_train) - top5.sum()
    if andere > 0:
        top5['Andere'] = andere
    wedges, texts, autotexts = ax.pie(
        top5.values,
        labels=top5.index,
        autopct='%1.0f%%',
        startangle=90,
        wedgeprops={'width': 0.6},
        textprops={'fontsize': 8},
    )
    ax.set_title('Sportarten-Verteilung')

    # Trainingsdauer-Verteilung
    ax = axes[2]
    dauer_clean = df_train['dauer_min'].dropna()
    dauer_clean = dauer_clean[dauer_clean < 300]  # < 5h
    ax.hist(dauer_clean, bins=40, color='#55A868', alpha=0.75, edgecolor='white')
    ax.axvline(dauer_clean.median(), color='#DD8452', lw=2,
               label=f'Median: {dauer_clean.median():.0f} min')
    ax.set_title('Trainingsdauer-Verteilung')
    ax.set_xlabel('Dauer (min)')
    ax.set_ylabel('Häufigkeit')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Trainings nach Wochentag ─────────────────────────────────
if not df_train.empty:
    wochentage_en = ['Monday','Tuesday','Wednesday',
                     'Thursday','Friday','Saturday','Sunday']
    wochentage_de = ['Mo','Di','Mi','Do','Fr','Sa','So']
    mapping = dict(zip(wochentage_en, wochentage_de))

    df_train['wochentag_de'] = df_train['wochentag'].map(mapping)
    wt_counts = df_train['wochentag_de'].value_counts().reindex(wochentage_de)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(wt_counts.index, wt_counts.values,
           color='#4C72B0', alpha=0.8, edgecolor='white')
    ax.set_title('Trainingseinheiten nach Wochentag')
    ax.set_xlabel('Wochentag')
    ax.set_ylabel('Anzahl Trainings')
    plt.tight_layout()
    plt.show()

## 6 – HRV: Überblick

In [ ]:
if not df_hrv.empty:
    df_hrv_m = df_hrv.set_index('datum').resample('ME').agg({
        'hrv_rmssd': 'mean',
        'hrv_sdnn' : 'mean',
    }).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('HRV – Überblick', fontsize=14, y=1.02)

    # RMSSD Zeitreihe
    ax = axes[0]
    ax.plot(df_hrv_m['datum'], df_hrv_m['hrv_rmssd'],
            color='#8172B2', lw=1.5, alpha=0.85)
    ax.fill_between(df_hrv_m['datum'], df_hrv_m['hrv_rmssd'],
                    alpha=0.15, color='#8172B2')
    ax.set_title('Ø RMSSD pro Monat (ms)')
    ax.set_ylabel('RMSSD (ms)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

    # RMSSD Saisonalität (nach Monat)
    ax = axes[1]
    df_hrv['monat'] = df_hrv['datum'].dt.month
    hrv_monat = df_hrv.groupby('monat')['hrv_rmssd'].mean()
    monate_de = ['Jan','Feb','Mär','Apr','Mai','Jun',
                 'Jul','Aug','Sep','Okt','Nov','Dez']
    ax.bar(range(1, 13), hrv_monat.reindex(range(1,13)),
           color='#8172B2', alpha=0.75, edgecolor='white')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(monate_de)
    ax.set_title('Ø RMSSD nach Monat (Saisonalität)')
    ax.set_ylabel('RMSSD (ms)')

    plt.tight_layout()
    plt.show()

    print(f"\nHRV-Statistik:")
    print(df_hrv[['hrv_rmssd','hrv_sdnn','hr_aus_ppi']].describe().round(2))
else:
    print("⚠️  Keine HRV-Daten verfügbar")

## 7 – Kombinierte Übersicht: Aktivitäts-Dashboard

In [ ]:
# ── KPI-Zusammenfassung ──────────────────────────────────────
print("╔" + "═" * 53 + "╗")
print("║  POLAR HEALTH ANALYTICS – KPI-ÜBERSICHT            ║")
print("╠" + "═" * 53 + "╣")

if not df_act.empty:
    jahre    = (df_act['datum'].max() - df_act['datum'].min()).days / 365.25
    schritte = df_act['schritte'].mean()
    schlaf   = df_act['schlaf_stunden'].mean()
    print(f"║  Datenspanne      : {jahre:.1f} Jahre                    ║")
    print(f"║  Ø Schritte/Tag   : {schritte:>6,.0f}                       ║")
    print(f"║  Ø Schlafdauer    : {schlaf:.2f}h                       ║")

if not df_hr.empty:
    ruhepuls = df_hr['hr_ruhepuls'].mean()
    print(f"║  Ø Ruhepuls       : {ruhepuls:.1f} bpm                     ║")

if not df_train.empty:
    n_trainings = len(df_train)
    n_laeufe    = len(df_train[df_train['sport'] == 'RUNNING'])
    print(f"║  Trainings gesamt : {n_trainings:>5}                        ║")
    print(f"║  Läufe gesamt     : {n_laeufe:>5}                        ║")

if not df_hrv.empty:
    rmssd = df_hrv['hrv_rmssd'].mean()
    print(f"║  Ø HRV RMSSD      : {rmssd:.1f} ms                      ║")

print("╚" + "═" * 53 + "╝")

db.schliessen()
print("\n✅ Exploration abgeschlossen – weiter mit 03_analysis.ipynb")